# SC4001 Phase 3, DAVIS DANN on two cold-target partitions

Accelerator must be **GPU (T4 x2 or P100)**. `src/train_dann.py` refuses CPU by design.

Code, processed tables and the 114-kinase external pool come from the attached dataset
`isaacwjk21/sc4001-project`. No internet and no installs are needed.

## Why two partitions rather than one or four

The Phase 2 split-seed sweep on KIBA showed that cold-target difficulty depends far more on
which genes land in the test set than on the training seed: MSE ran 0.352 (s4001) to 0.533
(s4004) across four gene partitions, a 0.18 swing, while seed scatter within a partition was
only 0.012 to 0.042. A DANN result measured on a single partition could therefore be an
artifact of a kind split.

Running all four partitions would cost four times the quota. Running the easiest and the
hardest instead brackets the range: if DANN helps at both ends, the conclusion holds across
everything in between; if it helps only on the kind split, that is exactly the lucky-split
caveat made visible. s4001 and s4004 were the extremes measured on KIBA, and the same two
split columns are reused here so the two datasets stay comparable.

## Grid

1. DANN on `split_cold_target` (s4001), seeds 4001/4002/4003, `--lambda-max 1.0`.
2. Phase 2 baseline on `split_cold_target_s4004`, seeds 4001/4002/4003. DAVIS has no baseline
   on this partition yet, and without it the DANN number on s4004 has nothing to be compared
   against.
3. DANN on `split_cold_target_s4004`, seeds 4001/4002/4003, `--lambda-max 1.0`.

Baselines for s4001 already exist from Phase 2, so they are not repeated.

Each run prints its own wall time. The first DANN run is also the first time this code has
touched a GPU, so its timing replaces the assumed 1.7x domain-branch overhead with a
measurement. Outputs go to `/kaggle/working/outputs` and survive only if this version is saved.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# Phase 3 needs the external kinase pool and the extra split columns, not just the Phase 2 tables.
required = ["external_kinases.csv", "external_tokens_davis.npz", "external_tokens_kiba.npz",
            "davis_extra_splits.csv", "kiba_extra_splits.csv"]
present = {p.name for p in (project / "data" / "processed").iterdir()}
missing = [name for name in required if name not in present and name + ".gz" not in present]
if missing:
    raise SystemExit("Dataset is missing Phase 3 inputs: " + str(missing))
print("phase 3 inputs OK")


In [ ]:
# Smoke the DANN entry point before spending quota: it has never run on a GPU.
smoke = subprocess.run(
    [sys.executable, "src/train_dann.py", "--dataset", "davis",
     "--split", "split_cold_target", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("DANN smoke run failed with code " + str(smoke.returncode)
                     + "; not starting the grid.")
for stale in Path("/kaggle/working/outputs").rglob("*-smoke"):
    shutil.rmtree(stale, ignore_errors=True)
print("smoke OK")


In [ ]:
# Each entry is (script, dataset, split, seed, extra flags).
# Order matters: the DANN runs on s4001 come first so the earliest wall times measure the
# domain-branch overhead, and so a partial kernel still yields one complete comparison.
GRID = (
    [("src/train_dann.py", "davis", "split_cold_target", seed, []) for seed in (4001, 4002, 4003)]
    + [("src/train.py", "davis", "split_cold_target_s4004", seed, []) for seed in (4001, 4002, 4003)]
    + [("src/train_dann.py", "davis", "split_cold_target_s4004", seed, []) for seed in (4001, 4002, 4003)]
)

started = time.time()
status = []
for script, dataset, split, seed, extra in GRID:
    t0 = time.time()
    label = ("DANN " if "dann" in script else "base ") + dataset + " " + split + " seed" + str(seed)
    print()
    print("===== " + label + " " + " ".join(extra) + " =====", flush=True)
    run = subprocess.run(
        [sys.executable, script, "--dataset", dataset, "--split", split,
         "--seed", str(seed), "--resume"] + extra,
        cwd=project, env=env, check=False,
    )
    elapsed = time.time() - t0
    status.append((label, run.returncode, elapsed))
    print("----- rc=" + str(run.returncode) + " wall=" + format(elapsed / 60, ".1f")
          + " min -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for label, rc, elapsed in status:
    print("%-52s rc=%d %6.1f min" % (label, rc, elapsed / 60))
failed = [row for row in status if row[1] != 0]
print("FAILED RUNS:", failed if failed else "none")


In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    phase = "phase3" if "phase3" in str(path) else "phase2"
    rows.append((phase, payload["split"], payload["seed"], payload.get("lambda_max"),
                 model["mse"], model["ci"], payload["best_epoch"], payload["total_seconds"]))

print("=" * 100)
print("%-7s %-26s %5s %7s %9s %8s %8s %9s"
      % ("phase", "split", "seed", "lambda", "mse", "ci", "best_ep", "minutes"))
for phase, split, seed, lam, mse, ci, best_ep, secs in sorted(rows):
    lam_text = "    n/a" if lam is None else "%7.2f" % lam
    print("%-7s %-26s %5d %s %9.4f %8.4f %8d %9.1f"
          % (phase, split, seed, lam_text, mse, ci, best_ep, secs / 60))
